# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts before building the app.

## Initialization

In [ ]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from pydantic import BaseModel
import os
import asyncio

load_dotenv(override=True)

def assertKeyExists(key :str) -> str:
    value = os.getenv(key)
    assert value, f"{key} not set"
    return value

openai_api_key = assertKeyExists("OPENAI_API_KEY")
google_api_key = assertKeyExists("GOOGLE_API_KEY")
grok_api_key = assertKeyExists("GROK_API_KEY")

print("API keys loaded ✔")

API keys loaded ✔


In [7]:
pushover_user = assertKeyExists("PUSHOVER_USER")
pushover_token = assertKeyExists("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [8]:
high_effort_model = "gpt-5.6-sol"
default_model = "gpt-5.6-luna"
low_effort_model = "gpt-5.6-terra"

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

GROK_BASE_URL = "https://api.x.ai/v1"
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
grok_model = OpenAIChatCompletionsModel(model="grok-4.5", openai_client=grok_client)


## Basic agent

A starting point with minimal agent to sanity-check the setup and test basic behavior.

In [10]:
from agents import Agent, Runner

agent = Agent(
    name="Meal Planner",
    instructions=(
        "You are a meal planning assistant. Suggest simple meals with minimal amounts of preparation that reheat well."
    ),
    model=default_model,
)
with trace("Basic Meal Plan"):
    result = await Runner.run(agent, "Suggest three vegetarian dinners for this week.")
    print(result.final_output)

1. **Vegetable chickpea curry with rice** — Simmer canned chickpeas, frozen vegetables, curry paste, and coconut milk. Reheats well for several days.

2. **Baked vegetable pasta** — Mix pasta with marinara, spinach, zucchini, and mozzarella; bake until bubbly. Great as leftovers.

3. **Black bean and sweet potato chili** — Combine canned black beans, canned tomatoes, diced sweet potato, corn, and chili spices in one pot. Reheats and freezes well.
